# Population-Based Optimization in Julia

This notebook introduces three important population-based optimization methods:

1. **Genetic Algorithms (GA)**
2. **Differential Evolution (DE)**
3. **Particle Swarm Optimization (PSO)**

These methods maintain a collection of candidate solutions and update the population iteratively.

They are especially useful when:

- derivatives are unavailable;
- the search space is non-convex or multimodal;
- local methods are likely to get trapped;
- the objective function is difficult to analyze analytically.

## Learning objectives

By the end of this notebook, you should be able to:

1. Explain the main components of a genetic algorithm.
2. Implement initialization, selection, crossover, and mutation.
3. Investigate how different GA operators affect performance.
4. Explain the mutation-and-recombination mechanism of Differential Evolution.
5. Explain the role of personal and global best solutions in PSO.
6. Compare GA, DE, and PSO on the same objective function.
7. Evaluate stochastic population-based algorithms using repeated runs.


## Packages


In [1]:
using Distributions 
import Random: seed!
import LinearAlgebra: norm

In [2]:
using StatsBase, Random, Distributions, Plots, LinearAlgebra

# Part I — Genetic Algorithms

A **genetic algorithm** is inspired by biological evolution.

A population of candidate solutions evolves through repeated application of:

1. **Initialization**
2. **Evaluation**
3. **Selection**
4. **Crossover**
5. **Mutation**
6. **Replacement**

For a minimization problem, individuals with lower objective values are generally considered better.

The balance between **exploration** and **exploitation** is controlled by the choice of operators and their parameters.


## 1. Population initialization

The first step is to generate an initial population.

If the search variables are bounded by

$$
a_j \le x_j \le b_j,
$$

a common approach is uniform random initialization.

For a population of size $m$, each individual is sampled independently from the search region.

A diverse initial population helps the algorithm explore different regions of the search space.


In [3]:
function rand_population_uniform(m, a, b)
    d = length(a)
    return [a + rand(d) .* (b - a) for i in 1:m]
end

function rand_population_normal(m, μ, Σ)
    D = MvNormal(μ, Σ)
    return [rand(D) for i in 1:m]

end

function rand_population_cauchy(m, μ, σ)
    n = length(μ)
    return [[rand(Cauchy(μ[j], σ[j])) for j in 1:n] for i in 1:m]
end

rand_population_binary(m, n) = [bitrand(n) for i in 1:m]

rand_population_binary (generic function with 1 method)

### Exercise 1 — Inspect the initial population

Generate populations of sizes

$$
m\in\{10,50,100\}
$$

for a two-dimensional search space

$$
[-5,5]^2.
$$

For each population:

1. create a scatter plot;
2. compute the mean of each coordinate;
3. compute the minimum and maximum coordinate values;
4. comment on how population size affects coverage of the search space.


In [4]:
# Exercise 1

# Generate several populations using rand_population_uniform.
# Plot them and compute simple summary statistics.


## 2. Selection

Selection determines which individuals are more likely to contribute to the next generation.

A good selection mechanism should:

- favor high-quality solutions;
- preserve enough diversity;
- avoid premature convergence.

The notebook defines a hierarchy of selection methods so different strategies can be tested with the same GA implementation.


In [5]:
abstract type SelectionMethod end
struct TruncationSelection <: SelectionMethod
    k # top k to keep
end
function select(t::TruncationSelection, y)
    p = sortperm(y)
    return [p[rand(1:t.k, 2)] for i in y]

end
struct TournamentSelection <: SelectionMethod
    k
end
function select(t::TournamentSelection, y)
    getparent() = begin
        p = randperm(length(y))
        p[argmin(y[p[1:t.k]])]
    end
    return [[getparent(), getparent()] for i in y]
end
struct RouletteWheelSelection <: SelectionMethod end
function select(::RouletteWheelSelection, y)
    y = maximum(y) .- y
    cat = Categorical(normalize(y, 1))
    return [rand(cat, 2) for i in y]
end

select (generic function with 3 methods)

In [6]:


# Define probabilities for each category
probabilities = [0.1, 0.3, 0.6]

# Create a categorical distribution
cat = Categorical(probabilities)

# Sample from the categorical distribution
samples = rand(cat,10)
cat, samples

(Categorical{Float64, Vector{Float64}}(support=Base.OneTo(3), p=[0.1, 0.3, 0.6]), [3, 2, 3, 3, 3, 2, 2, 3, 1, 3])

In [7]:
cat

Categorical{Float64, Vector{Float64}}(support=Base.OneTo(3), p=[0.1, 0.3, 0.6])

### Exercise 2 — Understand selection probabilities

Using a small population of five individuals with objective values

```julia
fitness = [1.0, 2.0, 4.0, 8.0, 16.0]
```

investigate the selection mechanism in the notebook.

1. Determine which individuals are favored.
2. Draw 1000 selections.
3. Count how often each individual is selected.
4. Plot the observed selection frequencies.

**Question:** Does the empirical result agree with the intended selection probabilities?


In [8]:
# Exercise 2

fitness = [1.0, 2.0, 4.0, 8.0, 16.0]

# Draw repeated selections and count the frequencies.


5-element Vector{Float64}:
  1.0
  2.0
  4.0
  8.0
 16.0

## 3. Crossover

Crossover combines information from two parent solutions to create offspring.

For real-valued optimization, crossover may combine coordinates directly or create weighted combinations of the parents.

The purpose is to exploit useful information already present in the population while still generating new candidate solutions.


In [9]:
abstract type CrossoverMethod end
struct SinglePointCrossover <: CrossoverMethod end

function crossover(::SinglePointCrossover, a, b)
    i = rand(1:length(a))
    return vcat(a[1:i], b[i+1:end])

end
struct TwoPointCrossover <: CrossoverMethod end
function crossover(::TwoPointCrossover, a, b)
    n = length(a)
    i, j = rand(1:n, 2)
    if i > j
        (i, j) = (j, i)
    end
    return vcat(a[1:i], b[i+1:j], a[j+1:n])
end
struct UniformCrossover <: CrossoverMethod end
function crossover(::UniformCrossover, a, b)
    child = copy(a)
    for i in 1:length(a)
        if rand() < 0.5
            child[i] = b[i]
        end
    end
    return child
end

struct InterpolationCrossover <: CrossoverMethod
    λ
end

crossover(C::InterpolationCrossover, a, b) = (1 - C.λ) * a + C.λ * b

crossover (generic function with 4 methods)

### Exercise 3 — Visualize crossover

Choose two two-dimensional parent vectors, for example

$$
p_1=(-3,2), \qquad p_2=(4,-1).
$$

Apply each crossover operator available in the notebook several times.

Plot:

- the two parents;
- the resulting offspring.

**Question:** Which crossover operator produces offspring only between the parents, and which may generate greater variation?


In [10]:
# Exercise 3

p1 = [-3.0, 2.0]
p2 = [4.0, -1.0]

# Apply the crossover operators and visualize the offspring.


2-element Vector{Float64}:
  4.0
 -1.0

## 4. Mutation

Mutation introduces random variation into the population.

Without mutation, the population can lose diversity and converge too early.

A mutation operator should perturb solutions enough to support exploration, but excessive mutation can destroy useful structure.

The implementation below defines mutation methods independently from selection and crossover, which makes it easy to compare operator combinations.


In [11]:
abstract type MutationMethod end
struct BitwiseMutation <: MutationMethod
    λ
end
function mutate(M::BitwiseMutation, child)
    return [rand() < M.λ ? !v : v for v in child]
end
struct GaussianMutation <: MutationMethod
    σ
end
function mutate(M::GaussianMutation, child)
    return child + randn(length(child)) * M.σ
end

mutate (generic function with 2 methods)

### Exercise 4 — Effect of mutation strength

Choose one individual and apply the mutation operator 500 times.

If the mutation method has a parameter controlling perturbation magnitude, test several values.

For each setting:

1. compute the average distance between the original and mutated solution;
2. plot the mutated solutions;
3. describe how mutation strength affects exploration.


In [12]:
# Exercise 4

x = [1.0, 1.0]

# Repeatedly mutate x and analyze the displacement.


2-element Vector{Float64}:
 1.0
 1.0

## 5. Complete genetic algorithm

The complete GA combines the previously defined operators.

At every generation, the algorithm evaluates the population and creates a new population using the selected operators.

Because the algorithm is stochastic, performance should be assessed over several independent runs rather than from a single execution.


In [13]:
function genetic_algorithm(f, population, max_iter, selection, crossover_, mutation)
    m = length(population)
    n = length(population[1])
    y = [f(population[i]) for i in 1:m]
    for i in 1:max_iter
        parents = select(selection, y)
        
        children = [crossover(crossover_, population[p[1]], population[p[2]]) for p in parents]
        children = [mutate(mutation, c) for c in children]
        children_y = [f(c) for c in children]
        for j in 1:m
            if children_y[j] < y[j]
                y[j] = children_y[j]
                population[j] = children[j]
            end
        end
        # population = children
        # y = children_y
        @show minimum(y)
    end
    return population[argmin(y)]
    
end

genetic_algorithm (generic function with 1 method)

In [14]:

seed!(0) # set random seed for reproducible results
f = x -> norm(x)
m = 100 # population size
k_max = 100 # number of iterations
population = rand_population_uniform(m, [-300, -300], [300, 300])
S = TruncationSelection(10) # select top 10
C = SinglePointCrossover()
M = GaussianMutation(0.5) # small mutation rate
x = genetic_algorithm(f, population, k_max, S, C, M)
@show x

minimum(y) = 28.187204656076105
minimum(y) = 27.65186117747279
minimum(y) = 26.878825735583185
minimum(y) = 25.62665494435819
minimum(y) = 24.394822906725494
minimum(y) = 23.380636095768946
minimum(y) = 22.335950063225496
minimum(y) = 21.35383619168129
minimum(y) = 20.602741672359056
minimum(y) = 19.576559546891126
minimum(y) = 17.760488789107804
minimum(y) = 16.767634923776555
minimum(y) = 15.950584920471282
minimum(y) = 15.184344402731929
minimum(y) = 14.249021357683917
minimum(y) = 13.471475487688327
minimum(y) = 12.5556887636712
minimum(y) = 11.650167731242947
minimum(y) = 10.445343680507769
minimum(y) = 9.189168572077437
minimum(y) = 8.123911173826386
minimum(y) = 6.596026736429999
minimum(y) = 5.569042278265519
minimum(y) = 4.7084239097176885
minimum(y) = 3.8941381472617644
minimum(y) = 3.160249277326202
minimum(y) = 2.5588376803684616
minimum(y) = 1.4856592352347142
minimum(y) = 0.4369218680984593
minimum(y) = 0.05449192075204436
minimum(y) = 0.05449192075204436
minimum(y) = 0.0

2-element Vector{Float64}:
 -0.0004397464784187538
 -0.0032156600189333917

### Exercise 5 — Test the GA on the Michalewicz function

The Michalewicz function is a common multimodal benchmark:

$$
f(x)
=
-\sum_{i=1}^n
\sin(x_i)
\left[
\sin\left(\frac{i x_i^2}{\pi}\right)
\right]^{2m},
$$

usually with

$$
0\le x_i\le \pi.
$$

Implement the two-dimensional version and run the GA.

1. Plot the objective function as a contour plot.
2. Run the GA from a random population.
3. Report the best solution and best objective value.
4. Plot the best-so-far objective value by generation.


In [15]:
# Exercise 5

function michalewicz(x; m=10)
    # Your implementation here
end

# Initialize a population in [0, π]^2 and run the GA.


michalewicz (generic function with 1 method)

### Exercise 6 — Repeated GA runs

Write a function

```julia
run_ga_experiment(...)
```

that runs the GA `NRuns` times and stores the best objective value obtained in each run.

Use at least 20 runs.

Compute:

- mean;
- standard deviation;
- median;
- minimum;
- maximum.

Create a boxplot or histogram of the final best values.


In [16]:
# Exercise 6

function run_ga_experiment(NRuns)
    # Run the GA repeatedly and return the best objective values.
end


run_ga_experiment (generic function with 1 method)

### Exercise 7 — Compare selection methods

Using the same objective function, population size, crossover, mutation, and number of generations:

1. test all selection methods implemented in the notebook;
2. perform multiple independent runs for each method;
3. compare mean and standard deviation of the final best objective value;
4. compare convergence curves.

**Question:** Is one selection mechanism consistently better?


In [17]:
# Exercise 7

# Compare all available selection methods using repeated runs.


### Exercise 8 — Compare crossover operators

Choose the selection method that performed best in the previous exercise.

Now compare all applicable crossover operators.

Keep every other parameter unchanged.

Use repeated runs and summarize the results statistically.


In [18]:
# Exercise 8

# Compare crossover operators.


### Exercise 9 — Compare mutation operators

Using the best selection and crossover settings found above, compare the available mutation strategies.

In addition to the final objective value, examine population diversity.

One simple diversity measure is the average Euclidean distance from individuals to the population mean.


In [19]:
# Exercise 9

# Compare mutation methods and optionally compute population diversity.


# Part II — Differential Evolution

Differential Evolution is a population-based method designed particularly for continuous optimization.

Unlike a standard GA, DE creates new candidate solutions using **scaled differences between population members**.

A typical mutation step is

$$
v_i
=
x_{r_1}
+
F(x_{r_2}-x_{r_3}),
$$

where $r_1,r_2,r_3$ are distinct indices.

This automatically adapts the scale of the search to the current population.

After mutation, DE usually performs crossover and then greedy selection between the trial solution and the current individual.


In [20]:

function differential_evolution(f, population, k_max; p=0.5, w=1)
    n, m = length(population[1]), length(population)
    @show n,m
    for k in 1:k_max
        for (k, x) in enumerate(population)
            @show k,x
            a, b, c = sample(population,
                Weights([j != k for j in 1:m]), 3, replace=false)
            
            z = a + w * (b - c)
            j = rand(1:n)
            x′ = [i == j || rand() < p ? z[i] : x[i] for i in 1:n]
            if f(x′) < f(x)
                x[:] = x′
            end

        end
        @show f(population[argmin(f.(population))])
    end
    return population[argmin(f.(population))]
end

differential_evolution (generic function with 1 method)

In [21]:
function rand_population_uniform(m, a, b)
    d = length(a)
    return [a + rand(d) .* (b - a) for i in 1:m]
end

function rand_population_normal(m, μ, Σ)
    D = MvNormal(μ, Σ)
    return [rand(D) for i in 1:m]

end

function rand_population_cauchy(m, μ, σ)
    n = length(μ)
    return [[rand(Cauchy(μ[j], σ[j])) for j in 1:n] for i in 1:m]
end

rand_population_binary(m, n) = [bitrand(n) for i in 1:m]

rand_population_binary (generic function with 1 method)

In [35]:
f = x -> norm(x)
m = 100 # population size
k_max = 100 # number of iterations
population = rand_population_uniform(m, [-300, -300], [300, 300])

best = differential_evolution(f, population,k_max)

2-element Vector{Float64}:
 1.0463751038969349e-6
 1.1764126384150586e-7

### Exercise 10 — Inspect the DE mutation mechanism

Generate a small two-dimensional population.

Choose three distinct individuals $x_{r_1},x_{r_2},x_{r_3}$ and compute

$$
v=x_{r_1}+F(x_{r_2}-x_{r_3})
$$

for several values of

$$
F\in\{0.2,0.5,1.0,1.5\}.
$$

Plot the three source individuals and the resulting mutant vectors.

**Question:** How does $F$ affect the search radius?


In [24]:
# Exercise 10

# Create a small population and visualize DE mutation for several F values.


### Exercise 11 — DE parameter study

Use the DE implementation on a benchmark function such as Ackley or Rosenbrock.

Investigate:

- crossover probability `p`;
- differential weight `w`.

For example, test

$$
p\in\{0.2,0.5,0.8\}
$$

and

$$
w\in\{0.4,0.8,1.2\}.
$$

Run every combination several times and store the final best objective value.

Create a heatmap or table summarizing the mean performance.


In [25]:
# Exercise 11

p_values = [0.2, 0.5, 0.8]
w_values = [0.4, 0.8, 1.2]

# Run a parameter study for Differential Evolution.


3-element Vector{Float64}:
 0.4
 0.8
 1.2

# Part III — Particle Swarm Optimization

Particle Swarm Optimization models a population as a swarm of moving particles.

Each particle has:

- a current position $x_i$;
- a velocity $v_i$;
- its personal best position $p_i$;
- access to the global best position $g$.

A standard velocity update is

$$
v_i
\leftarrow
w v_i
+
c_1 r_1(p_i-x_i)
+
c_2 r_2(g-x_i),
$$

followed by

$$
x_i\leftarrow x_i+v_i.
$$

The three main components are:

- **inertia**: continue moving in the previous direction;
- **cognitive component**: return toward the particle's own best solution;
- **social component**: move toward the swarm's best solution.


In [26]:
mutable struct Particle
    x
    v
    x_best

end

In [27]:
function rand_population_uniform_particles(m, a, b)
    d = length(a)
    population = [Particle(a + rand(d) .* (b - a), zeros(d), zeros(d)) for i in 1:m]
    for p in population
        p.x_best = p.x
    end
    return population
   
end

rand_population_uniform_particles (generic function with 1 method)

In [28]:
function particle_swarm_optimization(f, population, k_max; w=1, c1=1, c2=1)
    n = length(population[1].x)
    x_best, y_best = copy(population[1].x_best), Inf
    for P in population
        y = f(P.x)
        if y < y_best
            x_best[:], y_best = P.x, y
        end
    end
    for k in 1:k_max
        for P in population
            r1, r2 = rand(n), rand(n)
            P.x += P.v
            P.v = w * P.v + c1 * r1.*(P.x_best - P.x) +
                            c2 * r2.*(x_best - P.x)
            y = f(P.x)
            if y < y_best
                x_best[:], y_best = P.x, y
            end
            if y < f(P.x_best)
                P.x_best[:] = P.x
            end

        end
    end
    return y_best, x_best
end

particle_swarm_optimization (generic function with 1 method)

In [29]:
f = x -> norm(x)
m = 100 # population size
k_max = 500 # number of iterations
population = rand_population_uniform_particles(m, [-300, -300,-300,-300], [300, 300,300,300])
best = particle_swarm_optimization(f, population,k_max; w= 0.5)

(6.1900136888067134e-77, [5.180251019782694e-77, 1.3683976063823446e-77, -2.8337304976769627e-77, -1.25647455089126e-77])

### Exercise 12 — Interpret the PSO velocity update

For one particle, choose:

```julia
x = [2.0, -1.0]
v = [0.5, 0.2]
pbest = [1.0, 0.0]
gbest = [0.0, 0.0]
```

Using fixed random values such as `r1 = 0.4` and `r2 = 0.7`, compute one velocity update by hand for

$$
w=0.8,\quad c_1=1.5,\quad c_2=1.5.
$$

Then verify your calculation in Julia.


In [30]:
# Exercise 12

x = [2.0, -1.0]
v = [0.5, 0.2]
pbest = [1.0, 0.0]
gbest = [0.0, 0.0]

# Compute one PSO velocity and position update.


2-element Vector{Float64}:
 0.0
 0.0

### Exercise 13 — PSO parameter study

Investigate the roles of $w$, $c_1$, and $c_2$.

Try settings representing:

1. strong inertia;
2. strong cognitive attraction;
3. strong social attraction;
4. balanced coefficients.

Run each configuration several times on the same benchmark function.

Compare:

- convergence speed;
- final objective value;
- variability between runs.

**Question:** What behavior do you observe when the social component is much stronger than the cognitive component?


In [31]:
# Exercise 13

# Define several PSO parameter configurations and compare them.


# Part IV — Comparing GA, DE, and PSO

All three algorithms are population-based stochastic optimizers, but they generate search movements differently:

| Method | Main search mechanism |
|---|---|
| Genetic Algorithm | Selection + crossover + mutation |
| Differential Evolution | Differences between population vectors |
| Particle Swarm Optimization | Velocity based on personal and global experience |

A fair comparison should use:

- the same objective function;
- the same search bounds;
- comparable population sizes;
- comparable function-evaluation budgets;
- multiple independent runs.


### Exercise 14 — Benchmark GA, DE, and PSO

Choose one benchmark function and compare all three algorithms.

Use at least 20 independent runs for each method.

Store:

- method;
- run number;
- best objective value;
- number of iterations;
- number of function evaluations.

Create a `DataFrame` and compute grouped statistics.

Then create at least two plots, for example:

- boxplots of final best values;
- mean convergence curves;
- success rates;
- function evaluations required to reach a target value.

**Discussion questions**

1. Which method produces the best average result?
2. Which is the most consistent?
3. Which converges fastest?
4. Does the ranking change if function evaluations, rather than iterations, are used as the computational budget?


In [32]:
# Exercise 14

# using DataFrames

# Build a benchmarking experiment for GA, DE, and PSO.


## Reproducibility

Population-based algorithms are stochastic, so reproducibility is important.

Use

```julia
using Random
Random.seed!(1234)
```

when debugging or reproducing a particular experiment.

For performance comparisons, however, use multiple independent seeds and report both central tendency and variability.


### Exercise 15 — Robustness across seeds

Choose your preferred benchmark function.

Run GA, DE, and PSO using the same set of at least 20 random seeds.

For each seed, record the best final objective value for each method.

This creates a **paired experiment**, because every method is evaluated under the same seed set.

Compare:

- mean;
- median;
- standard deviation;
- paired differences between methods.

Optional: apply a paired non-parametric test such as the Wilcoxon signed-rank test.


In [33]:
# Exercise 15

seeds = 1:20

# Run all algorithms using the same seed set and compare paired results.


1:20

# Summary

This notebook introduced three major population-based optimization methods.

### Genetic Algorithms

GA evolves a population using:

- selection;
- crossover;
- mutation.

Its behavior depends strongly on the chosen operators.

### Differential Evolution

DE creates new candidates using differences between population members. This gives it a natural adaptive search scale for continuous problems.

### Particle Swarm Optimization

PSO moves particles according to:

- inertia;
- personal experience;
- social information.

The algorithms share an important feature: their output is stochastic. Therefore, reliable conclusions require **multiple independent runs** and careful control of computational budgets.

## Optional challenge

Choose three benchmark functions with different characteristics:

1. one convex/unimodal function;
2. one narrow-valley function;
3. one multimodal function.

Run GA, DE, and PSO on all three.

Create a final performance table ranking the methods separately for each problem.

Discuss whether one optimizer can be declared universally best.


In [34]:
# Optional challenge
